In [ ]:
!pip install pyculiarity

In [ ]:
import pandas as pd
import numpy as np
import math
import os, sys, gc, time, warnings, pickle, psutil, random

from tqdm import tqdm
from math import ceil

In [ ]:
grid_df = pd.read_parquet('train_val.parquet')
grid_df = grid_df.dropna(subset=['demand'])

In [ ]:
grid_df

,id,item_id,store_id,state_id,dept_id,cat_id,d,demand,lag_t7,rolling_mean_lag7_w7,...,wday,month,year,event_name_1,event_name_2,snap_CA,snap_TX,snap_WI,sell_price,disaster_severity_flag
0,HOBBIES_1_001_CA_1,1438,1,1,4,2,1029,0.0,0.0,1.000000,...,7,11,2013,1,1,0,0,0,8.26,0
1,HOBBIES_1_002_CA_1,1439,1,1,4,2,1029,0.0,1.0,0.142857,...,7,11,2013,1,1,0,0,0,3.97,0
2,HOBBIES_1_003_CA_1,1440,1,1,4,2,1029,0.0,0.0,0.000000,...,7,11,2013,1,1,0,0,0,NaN,0
3,HOBBIES_1_004_CA_1,1441,1,1,4,2,1029,3.0,0.0,1.142857,...,7,11,2013,1,1,0,0,0,4.64,0
4,HOBBIES_1_005_CA_1,1442,1,1,4,2,1029,1.0,1.0,0.428571,...,7,11,2013,1,1,0,0,0,3.08,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27837365,FOODS_3_823_WI_3,1433,10,3,3,1,1941,1.0,3.0,0.714286,...,2,5,2016,1,1,0,0,0,2.98,0
27837366,FOODS_3_824_WI_3,1434,10,3,3,1,1941,0.0,0.0,0.000000,...,2,5,2016,1,1,0,0,0,2.48,0
27837367,FOODS_3_825_WI_3,1435,10,3,3,1,1941,2.0,1.0,1.142857,...,2,5,2016,1,1,0,0,0,3.98,0
27837368,FOODS_3_826_WI_3,1436,10,3,3,1,1941,0.0,1.0,1.000000,...,2,5,2016,1,1,0,0,0,1.28,0


In [ ]:
print(grid_df.columns)

Index(['id', 'item_id', 'store_id', 'state_id', 'dept_id', 'cat_id', 'd',
       'demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28',
       'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'wday',
       'month', 'year', 'event_name_1', 'event_name_2', 'snap_CA', 'snap_TX',
       'snap_WI', 'sell_price', 'disaster_severity_flag'],
      dtype='object')


In [ ]:
duplicate_id_list = grid_df[grid_df.duplicated('id', keep=False)]['id'].unique()

print(duplicate_id_list)

['HOBBIES_1_001_CA_1', 'HOBBIES_1_002_CA_1', 'HOBBIES_1_003_CA_1', 'HOBBIES_1_004_CA_1', 'HOBBIES_1_005_CA_1', ..., 'FOODS_3_823_WI_3', 'FOODS_3_824_WI_3', 'FOODS_3_825_WI_3', 'FOODS_3_826_WI_3', 'FOODS_3_827_WI_3']
Length: 30490
Categories (30490, object): ['FOODS_1_001_CA_1', 'FOODS_1_001_CA_2', 'FOODS_1_001_CA_3',
                             'FOODS_1_001_CA_4', ..., 'HOUSEHOLD_2_516_TX_3', 'HOUSEHOLD_2_516_WI_1',
                             'HOUSEHOLD_2_516_WI_2', 'HOUSEHOLD_2_516_WI_3']


In [ ]:
cal = pd.read_parquet('done_cal (1).parquet')
print(cal)

           date  wm_yr_wk  wday  month  year     d  event_name_1  \
0    2011-01-29     11101     1      1  2011     1             1   
1    2011-01-30     11101     2      1  2011     2             1   
2    2011-01-31     11101     3      1  2011     3             1   
3    2011-02-01     11101     4      2  2011     4             1   
4    2011-02-02     11101     5      2  2011     5             1   
...         ...       ...   ...    ...   ...   ...           ...   
1964 2016-06-15     11620     5      6  2016  1965             1   
1965 2016-06-16     11620     6      6  2016  1966             1   
1966 2016-06-17     11620     7      6  2016  1967             1   
1967 2016-06-18     11621     1      6  2016  1968             1   
1968 2016-06-19     11621     2      6  2016  1969            18   

      event_name_2  snap_CA  snap_TX  snap_WI  
0                1        0        0        0  
1                1        0        0        0  
2                1        0        0   

In [ ]:
cnt = 0
for instance in duplicate_id_list:
  cnt += 1
  if cnt % 1000 == 0:
    print(cnt)
  filtered_df = grid_df[grid_df.id == instance]

  sales_ts = pd.DataFrame({
     'timestamps': filtered_df['d'].values,
     'demand': filtered_df['demand'].values
  })
  sales_ts['d'] = sales_ts['timestamps'] # keep sales_ts['d'] as integer

  # Merge với calendar_df theo cột 'd'
  sales_ts = sales_ts.merge(cal[['d', 'date']], on='d', how='left')

  sales_ts['timestamps'] = sales_ts['date']
  sales_ts = sales_ts.drop(columns=['date'])

  sales_ts['EMA'] = sales_ts['demand'].ewm(span=2, adjust=False).mean()
  sales_ts['EMA'] = sales_ts['EMA'].apply(math.ceil)

  from statsmodels.tsa.seasonal import STL
  from scipy import stats
  import numpy as np

  # Phân tích chuỗi theo mùa vụ
  stl = STL(sales_ts['demand'], period=30)  # nếu data theo ngày, 7 là 1 tuần
  res = stl.fit()

  residual = res.resid

  # Áp dụng z-score để tìm outlier trong residual
  z = np.abs(stats.zscore(residual))
  threshold = 2.8  # ngưỡng z-score
  outliers = sales_ts[z > threshold]


  for idx, row in outliers.iterrows():
       grid_df.loc[(grid_df.id == instance) & (grid_df['d'] == row['d']), 'demand'] = row['EMA']

KeyboardInterrupt: 